# JEPA — Joint Embedding Predictive Architecture
## Tutorial Notebook 01: Concepts, Architecture & PyTorch Implementation

**Reference:** Yann LeCun, *"A Path Towards Autonomous Machine Intelligence"* (2022)  
**Paper:** Assran et al., *"Self-Supervised Learning from Images with a Joint-Embedding Predictive Architecture"* — I-JEPA (2023)

---

### What you will learn
1. What JEPA is and how it differs from contrastive and generative self-supervised learning
2. The three-component architecture: Context Encoder, Target Encoder, Predictor
3. Why the target encoder uses an **Exponential Moving Average (EMA)** of the context encoder
4. How masking strategies create the prediction task
5. A minimal PyTorch implementation of **I-JEPA** (image-domain JEPA)
6. The training loop with the JEPA loss
7. How to extend to **V-JEPA** (video) and **A-JEPA** (audio)

## 1. Motivation — Why JEPA?

### The Problem with Prior Self-Supervised Methods

| Approach | Example | Problem |
|----------|---------|------|
| **Generative** | Masked Autoencoders (MAE) | Must reconstruct pixel-level detail — wastes capacity on irrelevant texture |
| **Contrastive** | SimCLR, MoCo | Requires hand-crafted augmentation invariances; negative samples or special batching tricks |
| **Joint Embedding** | CLIP | Needs paired supervision (text–image); collapses without careful design |

### LeCun's Insight

LeCun argues that intelligent systems should learn by **predicting in representation space**, not pixel space. A world model should ask:

> *"Given this partial view of the world, what will the representation of the hidden part look like?"*

Not: *"What will the exact pixels look like?"*

This is the JEPA idea: **predict abstract embeddings, not raw data**.

```
Generative (MAE):  context patches → [predict] → target pixels      ← hard, wasteful
JEPA:              context patches → [predict] → target embeddings   ← abstract, efficient
```

## 2. The JEPA Architecture

JEPA has three learned components:

```
                       ┌──────────────────────────────────────────────────┐
                       │                  IMAGE / VIDEO                    │
                       └──────────────┬───────────────────────────────────┘
                                      │  patch tokenisation
                     ┌────────────────┴────────────────┐
                     │                                 │
              context patches                   target patches
              (visible region)                  (masked region)
                     │                                 │
             ┌───────▼───────┐                ┌────────▼──────────┐
             │  Context      │                │  Target Encoder   │
             │  Encoder  fθ  │                │  (EMA of fθ) fξ   │
             └───────┬───────┘                └────────┬──────────┘
                     │  z_context                      │  z_target  (stop-grad)
                     │                                 │
             ┌───────▼───────┐                         │
             │   Predictor   │─── predicts ───────────►│
             │     gφ        │   ŝ_target               │
             └───────────────┘                          │
                                                        │
                     Loss: L = || ŝ_target - z_target ||²
                           (in normalised embedding space)
```

### Key Design Choices

1. **EMA Target Encoder** — the target encoder is *not* trained by backpropagation. It is updated as a slow exponential moving average of the context encoder weights. This prevents representation collapse.

2. **Narrow Predictor** — the predictor `gφ` is intentionally *weaker* than the encoder. It can only succeed by learning an encoder that produces semantically useful representations. A powerful predictor would short-circuit learning.

3. **No Negative Samples, No Augmentation Invariance** — unlike contrastive methods, JEPA does not require carefully designed augmentations. The mask strategy alone creates the self-supervised signal.

4. **Stop Gradient on Target** — gradients are *not* propagated through the target encoder branch. Only the context encoder + predictor receive gradient updates.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy
import numpy as np
from typing import Tuple, List

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 3. Patch Embedding — Tokenising the Image

Like Vision Transformers (ViT), I-JEPA splits an image into non-overlapping **patches** and linearly projects each patch into a `d_model`-dimensional token.

For an image of size `H × W` with patch size `P × P`:
- Number of patches: `N = (H/P) × (W/P)`
- Each patch is flattened to `P² × C` (C = channels) then projected to `d_model`

A learnable positional embedding is added so the encoder knows where each patch came from.

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Splits an image into non-overlapping patches and projects each
    to a d_model-dimensional embedding vector.

    Args:
        img_size:   Square image side length (e.g. 224)
        patch_size: Square patch side length (e.g. 16)
        in_chans:   Input channels (3 for RGB)
        embed_dim:  Output embedding dimension
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        assert img_size % patch_size == 0, "Image size must be divisible by patch size"

        self.img_size    = img_size
        self.patch_size  = patch_size
        self.num_patches = (img_size // patch_size) ** 2   # e.g. (224/16)^2 = 196
        self.embed_dim   = embed_dim

        # A single Conv2d with kernel_size=stride=patch_size is equivalent to
        # extracting patches and applying a linear projection — efficient and standard.
        self.projection = nn.Conv2d(
            in_chans, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )

        # Learnable positional embeddings — one per patch position
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, C, H, W)
        Returns:
            tokens: (B, N, embed_dim)  where N = num_patches
        """
        B = x.shape[0]
        # Conv2d → (B, embed_dim, H/P, W/P)
        x = self.projection(x)
        # Flatten spatial dims and transpose → (B, N, embed_dim)
        x = x.flatten(2).transpose(1, 2)
        # Add positional embedding (broadcast over batch)
        x = x + self.pos_embed
        return x


# --- Quick shape check ---
patch_emb = PatchEmbedding(img_size=224, patch_size=16, in_chans=3, embed_dim=768)
dummy_img = torch.randn(2, 3, 224, 224)  # batch of 2 RGB images
tokens = patch_emb(dummy_img)
print(f"Input image shape:  {dummy_img.shape}")
print(f"Token output shape: {tokens.shape}")
print(f"Number of patches:  {patch_emb.num_patches}  (= (224/16)² = 196)")

## 4. Masking Strategy

I-JEPA uses a specific masking scheme designed to create a **non-trivial prediction task**:

- **Target blocks**: 4 large, possibly overlapping rectangular regions (≈15% of patches each)
- **Context region**: the complement of the union of all target blocks — what the context encoder sees

The large contiguous target blocks force the predictor to reason about **spatial structure**, not just local texture. This is unlike MAE's random per-patch masking.

```
Full image (14×14 patches for a 224×224 / patch=16 example):

  ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░
  ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░
  ░ ░ ▓ ▓ ▓ ▓ ░ ░ ░ ░ ░ ░ ░ ░   ← target block 1
  ░ ░ ▓ ▓ ▓ ▓ ░ ░ ░ ░ ░ ░ ░ ░
  ░ ░ ▓ ▓ ▓ ▓ ░ ░ ░ ░ ░ ░ ░ ░
  ░ ░ ░ ░ ░ ░ ░ ░ ░ █ █ █ ░ ░   ← target block 2
  ░ ░ ░ ░ ░ ░ ░ ░ ░ █ █ █ ░ ░
  ...

  ░ = context patch (visible to context encoder)
  ▓/█ = target patch (masked from context encoder, predicted by predictor)
```

In [ ]:
class IJEPAMaskGenerator:
    """
    Generates context and target patch masks for I-JEPA.

    Target masks: num_target_blocks contiguous rectangular blocks.
    Context mask: complement of the union of all target blocks,
                  further subsampled to context_scale of available patches.
    """
    def __init__(
        self,
        input_size: Tuple[int, int] = (14, 14),  # grid of patches (H/P, W/P)
        num_target_blocks: int = 4,
        target_scale: Tuple[float, float] = (0.15, 0.2),   # fraction of patches per block
        target_aspect_ratio: Tuple[float, float] = (0.75, 1.5),
        context_scale: Tuple[float, float] = (0.85, 1.0),  # fraction of non-target patches to keep
    ):
        self.grid_h, self.grid_w = input_size
        self.num_patches         = self.grid_h * self.grid_w
        self.num_target_blocks   = num_target_blocks
        self.target_scale        = target_scale
        self.target_aspect_ratio = target_aspect_ratio
        self.context_scale       = context_scale

    def _sample_block(self) -> List[int]:
        """Sample one rectangular target block; return sorted patch indices."""
        for _ in range(10):  # retry if block is out of bounds
            scale = np.random.uniform(*self.target_scale)
            area  = int(self.num_patches * scale)
            ar    = np.random.uniform(*self.target_aspect_ratio)
            h     = max(1, int(round(math.sqrt(area * ar))))
            w     = max(1, int(round(math.sqrt(area / ar))))
            if h <= self.grid_h and w <= self.grid_w:
                top  = np.random.randint(0, self.grid_h - h + 1)
                left = np.random.randint(0, self.grid_w - w + 1)
                indices = [
                    (top + i) * self.grid_w + (left + j)
                    for i in range(h) for j in range(w)
                ]
                return indices
        return []  # fallback (very rare)

    def __call__(self) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """
        Returns:
            context_mask:  1-D LongTensor of context patch indices
            target_masks:  List of 1-D LongTensors, one per target block
        """
        # Sample target blocks and union their indices
        target_indices_set = set()
        target_masks = []
        for _ in range(self.num_target_blocks):
            block = self._sample_block()
            target_indices_set.update(block)
            target_masks.append(torch.tensor(sorted(set(block)), dtype=torch.long))

        # Context = all patches NOT in any target block, then subsample
        all_indices     = set(range(self.num_patches))
        context_indices = sorted(all_indices - target_indices_set)
        # Subsample context to context_scale fraction
        keep = max(1, int(len(context_indices) * np.random.uniform(*self.context_scale)))
        context_indices = sorted(np.random.choice(context_indices, keep, replace=False).tolist())
        context_mask = torch.tensor(context_indices, dtype=torch.long)

        return context_mask, target_masks


# --- Demo ---
mask_gen = IJEPAMaskGenerator(input_size=(14, 14), num_target_blocks=4)
ctx_mask, tgt_masks = mask_gen()

print(f"Total patches:          196  (14×14)")
print(f"Context patches kept:   {len(ctx_mask)}")
print(f"Number of target masks: {len(tgt_masks)}")
print(f"Target block sizes:     {[len(m) for m in tgt_masks]}")
print(f"\nContext mask (first 10 indices): {ctx_mask[:10].tolist()}")
print(f"Target block 0 (first 10):       {tgt_masks[0][:10].tolist()}")

## 5. The Vision Transformer Encoder

Both the context encoder and target encoder share the **same ViT architecture** (the target encoder starts as a copy of the context encoder). We implement a minimal ViT encoder:

- **Multi-Head Self-Attention** (standard, no causal mask — full bidirectional)
- **MLP block** with GELU activation
- **Pre-norm** (LayerNorm before each sub-layer — modern standard)
- No CLS token needed (I-JEPA uses all patch tokens directly)

In [ ]:
class Attention(nn.Module):
    """Multi-Head Self-Attention with pre-normalisation."""
    def __init__(self, dim: int, num_heads: int = 8, qkv_bias: bool = True, attn_drop: float = 0.0):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5  # 1/√d_head for numerical stability

        # Fused QKV projection — one matrix multiply instead of three
        self.qkv  = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(attn_drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape   # batch, num_tokens, embed_dim

        # Project to Q, K, V and split heads
        qkv = self.qkv(x)                              # (B, N, 3*C)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)              # (3, B, heads, N, head_dim)
        q, k, v = qkv.unbind(0)                       # each: (B, heads, N, head_dim)

        # Scaled dot-product attention
        # PyTorch 2.0+ F.scaled_dot_product_attention uses Flash Attention automatically
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, heads, N, N)
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        # Weighted sum of values
        x = (attn @ v)                                 # (B, heads, N, head_dim)
        x = x.transpose(1, 2).reshape(B, N, C)        # (B, N, C)
        return self.proj(x)


class MLP(nn.Module):
    """Position-wise MLP with GELU activation (FFN in ViT)."""
    def __init__(self, in_features: int, hidden_features: int, drop: float = 0.0):
        super().__init__()
        self.fc1  = nn.Linear(in_features, hidden_features)
        self.act  = nn.GELU()
        self.fc2  = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.fc2(x))
        return x


class TransformerBlock(nn.Module):
    """One ViT encoder block: Pre-Norm → Attention + MLP."""
    def __init__(self, dim: int, num_heads: int, mlp_ratio: float = 4.0,
                 qkv_bias: bool = True, drop: float = 0.0, attn_drop: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = MLP(dim, int(dim * mlp_ratio), drop=drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))   # pre-norm + residual
        x = x + self.mlp(self.norm2(x))    # pre-norm + residual
        return x


class VisionTransformerEncoder(nn.Module):
    """
    Minimal ViT encoder used for both Context and Target encoders in I-JEPA.

    Accepts a subset of patch indices (the visible/context patches) and
    returns their contextualised embeddings.
    """
    def __init__(
        self,
        img_size: int = 224,
        patch_size: int = 16,
        in_chans: int = 3,
        embed_dim: int = 768,
        depth: int = 12,
        num_heads: int = 12,
        mlp_ratio: float = 4.0,
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_chans, embed_dim)
        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, mlp_ratio)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor, patch_indices: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x:             (B, C, H, W) input images
            patch_indices: 1-D LongTensor of patch positions to keep
                           (None = keep all patches)
        Returns:
            (B, len(patch_indices), embed_dim)  or  (B, N, embed_dim) if indices=None
        """
        tokens = self.patch_embed(x)               # (B, N, embed_dim)

        if patch_indices is not None:
            # Select only the specified patches — creates the masked view
            tokens = tokens[:, patch_indices, :]   # (B, |mask|, embed_dim)

        tokens = self.blocks(tokens)               # self-attention over visible patches only
        tokens = self.norm(tokens)
        return tokens


# --- Shape check ---
encoder = VisionTransformerEncoder(
    img_size=224, patch_size=16, embed_dim=384, depth=6, num_heads=6
)
img = torch.randn(2, 3, 224, 224)

# Full forward (no masking)
out_full = encoder(img)
print(f"Full forward output:    {out_full.shape}   (B=2, N=196, D=384)")

# Context-only forward (using a context mask)
ctx_mask, _ = mask_gen()
out_ctx = encoder(img, patch_indices=ctx_mask)
print(f"Context forward output: {out_ctx.shape}  (B=2, |ctx|, D=384)")

## 6. The Predictor

The predictor `gφ` takes as input:
1. **Context representations** — the output of the context encoder for visible patches
2. **Mask tokens** — learnable query vectors placed at target positions (like decoder queries in DETR)

It outputs a predicted representation for each target patch position.

The predictor is a **narrow** Transformer — typically much smaller than the encoder (fewer layers, smaller hidden dim). This forces the encoder to capture rich structure; the predictor just queries that structure.

```
Predictor input:
  [ctx_emb_1, ctx_emb_2, ..., ctx_emb_k,   ← context encoder outputs (at ctx positions)
   mask_tok + pos_emb_t1,                   ← learnable query at target position t1
   mask_tok + pos_emb_t2,                   ← learnable query at target position t2
   ...]

Predictor output (only target positions extracted):
  [ŝ_t1, ŝ_t2, ...]   → compared to z_t1, z_t2 from target encoder
```

In [ ]:
class IJEPAPredictor(nn.Module):
    """
    I-JEPA Predictor: a narrow Transformer that predicts target patch
    representations from context representations.

    Architecture:
      1. Project context embeddings from encoder_dim to predictor_dim
      2. Concatenate with learnable mask tokens (at target positions)
      3. Run through predictor_depth Transformer blocks
      4. Extract target-position outputs and project back to encoder_dim
    """
    def __init__(
        self,
        num_patches: int,     # total number of patches in the image (e.g. 196)
        encoder_dim: int,     # dimension of encoder output
        predictor_dim: int,   # internal width of predictor (narrower than encoder)
        depth: int = 6,       # number of Transformer blocks in predictor
        num_heads: int = 12,
    ):
        super().__init__()
        self.predictor_dim = predictor_dim

        # Project encoder dim → predictor dim
        self.input_proj  = nn.Linear(encoder_dim, predictor_dim)

        # Learnable positional embeddings for ALL patch positions
        # (shared between context and target, each position has a unique embedding)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, predictor_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # Learnable mask token — a single vector shared across all masked positions
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)

        # Narrow Transformer backbone
        self.blocks = nn.Sequential(*[
            TransformerBlock(predictor_dim, num_heads)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(predictor_dim)

        # Project predictor dim → encoder dim so we can compare with target encoder output
        self.output_proj = nn.Linear(predictor_dim, encoder_dim)

    def forward(
        self,
        context_emb: torch.Tensor,   # (B, |ctx|, encoder_dim)
        context_indices: torch.Tensor,  # 1-D LongTensor of context positions
        target_indices: torch.Tensor,   # 1-D LongTensor of target positions
    ) -> torch.Tensor:
        """
        Returns:
            predictions: (B, |target|, encoder_dim) — predicted target embeddings
        """
        B = context_emb.shape[0]

        # 1. Project context embeddings to predictor dimension
        ctx = self.input_proj(context_emb)  # (B, |ctx|, predictor_dim)

        # 2. Add positional embeddings for context positions
        ctx = ctx + self.pos_embed[:, context_indices, :]  # broadcast over B

        # 3. Build mask tokens for each target position
        #    Expand the single mask token to (B, |target|, predictor_dim)
        n_tgt = len(target_indices)
        mask_tokens = self.mask_token.expand(B, n_tgt, -1).clone()
        # Add positional embeddings for target positions
        mask_tokens = mask_tokens + self.pos_embed[:, target_indices, :]  # (B, |tgt|, pd)

        # 4. Concatenate context + mask tokens → full predictor sequence
        x = torch.cat([ctx, mask_tokens], dim=1)  # (B, |ctx|+|tgt|, predictor_dim)

        # 5. Run through Transformer blocks
        x = self.blocks(x)
        x = self.norm(x)

        # 6. Extract only the target-position outputs (last |tgt| tokens)
        predictions = x[:, len(context_indices):, :]    # (B, |tgt|, predictor_dim)

        # 7. Project back to encoder dimension for loss comparison
        return self.output_proj(predictions)             # (B, |tgt|, encoder_dim)


# --- Shape check ---
predictor = IJEPAPredictor(
    num_patches=196,
    encoder_dim=384,
    predictor_dim=192,   # narrower than encoder
    depth=6,
    num_heads=12,
)

dummy_ctx_emb = torch.randn(2, len(ctx_mask), 384)
preds = predictor(dummy_ctx_emb, ctx_mask, tgt_masks[0])
print(f"Predictor input  (context emb):  {dummy_ctx_emb.shape}")
print(f"Predictor output (predictions):  {preds.shape}")
print(f"  → predicts {len(tgt_masks[0])} target patches with dim=384")

## 7. Exponential Moving Average (EMA) Target Encoder

The target encoder is **never** updated by backpropagation. Instead, its weights are updated as an exponential moving average of the context encoder weights after every training step:

```
ξ ← τ · ξ + (1 - τ) · θ
```

where:
- `θ` = context encoder parameters (trained by gradient descent)
- `ξ` = target encoder parameters (updated by EMA only)
- `τ` = momentum coefficient, starts high (e.g. 0.996) and is annealed toward 1.0 during training

**Why does this prevent collapse?**

If both encoders were trained with backprop, the trivial solution (encode everything as the same constant vector) would produce zero loss. The EMA target encoder acts as a slowly-moving but non-trivial target — it is always slightly "ahead" of the context encoder, so the context encoder must genuinely track a moving target that reflects the recent history of learned representations.

This is similar to BYOL and MoCo, but here there are no augmentation invariances baked in.

In [ ]:
@torch.no_grad()
def update_ema(context_encoder: nn.Module, target_encoder: nn.Module, momentum: float):
    """
    Update target encoder weights as EMA of context encoder.

    ξ ← τ · ξ + (1 - τ) · θ

    Args:
        context_encoder: trained by gradient descent (θ)
        target_encoder:  updated by EMA only (ξ)
        momentum:        τ ∈ [0, 1], higher = slower target update
    """
    for param_q, param_k in zip(context_encoder.parameters(),
                                 target_encoder.parameters()):
        # param_k.data: target encoder weights  (ξ)
        # param_q.data: context encoder weights (θ)
        param_k.data = momentum * param_k.data + (1.0 - momentum) * param_q.data


def cosine_ema_schedule(base_momentum: float, final_momentum: float,
                         step: int, total_steps: int) -> float:
    """
    Cosine annealing schedule for EMA momentum τ.
    Starts at base_momentum (e.g. 0.996) and approaches final_momentum (e.g. 1.0).
    """
    return final_momentum - (final_momentum - base_momentum) * (
        math.cos(math.pi * step / total_steps) + 1
    ) / 2


# --- Demo: EMA update ---
ctx_enc = VisionTransformerEncoder(img_size=224, patch_size=16, embed_dim=384, depth=6, num_heads=6)
tgt_enc = copy.deepcopy(ctx_enc)      # target starts as an exact copy
tgt_enc.requires_grad_(False)         # target encoder: no gradients

# Simulate a weight change in the context encoder
with torch.no_grad():
    for p in ctx_enc.parameters():
        p.data += 0.01 * torch.randn_like(p.data)  # pretend SGD update happened

# Before EMA update
p_ctx = list(ctx_enc.parameters())[0]
p_tgt = list(tgt_enc.parameters())[0]
diff_before = (p_ctx.data - p_tgt.data).abs().mean().item()
print(f"Parameter diff BEFORE EMA update: {diff_before:.6f}")

update_ema(ctx_enc, tgt_enc, momentum=0.996)

p_tgt_new = list(tgt_enc.parameters())[0]
diff_after = (p_ctx.data - p_tgt_new.data).abs().mean().item()
print(f"Parameter diff AFTER  EMA update: {diff_after:.6f}")
print(f"  Target encoder moved slightly toward context encoder (τ=0.996 → slow update)")

## 8. The JEPA Loss Function

The I-JEPA loss is a **smooth L2 loss in normalised representation space**:

```
For each target block b and each patch position i in block b:

  L = (1/|B| × |b|) Σ_b Σ_i  || ŝ_i  -  sg(z_i) ||²
                                  ↑          ↑
                            predictor    target encoder
                            output       output (stop-grad)
```

- `sg(·)` = stop-gradient (the target encoder branch receives no gradients)
- Both `ŝ_i` and `z_i` are **L2-normalised** before computing the loss
- Averaging over all target blocks and patch positions in each batch

L2 normalisation prevents collapse to zero (you can't minimise norm of unit vectors).

In [ ]:
def jepa_loss(
    predictions: List[torch.Tensor],   # list of (B, |tgt_b|, D) — one per target block
    targets: List[torch.Tensor],        # list of (B, |tgt_b|, D) — from target encoder, stop-grad
) -> torch.Tensor:
    """
    Compute the I-JEPA loss: mean squared error in L2-normalised embedding space.

    Args:
        predictions: List of predictor outputs for each target block
        targets:     List of target encoder outputs for each target block (no grad)
    Returns:
        Scalar loss tensor
    """
    assert len(predictions) == len(targets)
    total_loss = 0.0
    total_patches = 0

    for pred, tgt in zip(predictions, targets):
        # L2-normalise along the embedding dimension
        pred_norm = F.normalize(pred, dim=-1)   # (B, |tgt|, D)
        tgt_norm  = F.normalize(tgt,  dim=-1)   # (B, |tgt|, D)

        # Mean squared error — equivalent to 2 - 2·cosine_similarity when normalised
        loss = F.mse_loss(pred_norm, tgt_norm, reduction='sum')
        total_loss    += loss
        total_patches += pred.shape[0] * pred.shape[1]  # B × |tgt|

    return total_loss / total_patches


# --- Demo loss with random tensors ---
B, D = 2, 384
dummy_preds   = [torch.randn(B, 20, D) for _ in range(4)]
dummy_targets = [torch.randn(B, 20, D) for _ in range(4)]
loss_val = jepa_loss(dummy_preds, dummy_targets)
print(f"JEPA loss (random initialisation): {loss_val.item():.4f}")
print(f"  (expect ~2.0 for orthogonal random unit vectors — MSE between them)")

# Perfect predictions should give loss ≈ 0
perfect_preds = [t.clone() for t in dummy_targets]
print(f"JEPA loss (perfect predictions):   {jepa_loss(perfect_preds, dummy_targets).item():.6f}")

## 9. Full I-JEPA Model

Assembling everything into one `IJEPA` module that wraps:
- Context encoder `fθ` (trained by backprop)
- Target encoder `fξ` (EMA copy, no grad)
- Predictor `gφ` (trained by backprop)

And provides a `forward()` method that:
1. Encodes context patches with `fθ`
2. Encodes target patches with `fξ` (no grad)
3. Predicts target embeddings with `gφ`
4. Returns predictions and targets for loss computation

In [ ]:
class IJEPA(nn.Module):
    """
    Complete I-JEPA model.

    Holds the context encoder, target encoder (EMA), and predictor.
    The target encoder is registered as a separate attribute and is
    excluded from optimizer parameter groups.
    """
    def __init__(
        self,
        img_size: int = 224,
        patch_size: int = 16,
        in_chans: int = 3,
        encoder_dim: int = 384,
        encoder_depth: int = 12,
        encoder_heads: int = 6,
        predictor_dim: int = 192,
        predictor_depth: int = 6,
        predictor_heads: int = 12,
        ema_momentum: float = 0.996,
    ):
        super().__init__()
        num_patches = (img_size // patch_size) ** 2

        # ── Context Encoder (trained) ─────────────────────────────────
        self.context_encoder = VisionTransformerEncoder(
            img_size, patch_size, in_chans,
            encoder_dim, encoder_depth, encoder_heads
        )

        # ── Target Encoder (EMA — no grad) ────────────────────────────
        self.target_encoder = copy.deepcopy(self.context_encoder)
        self.target_encoder.requires_grad_(False)

        # ── Predictor ─────────────────────────────────────────────────
        self.predictor = IJEPAPredictor(
            num_patches, encoder_dim,
            predictor_dim, predictor_depth, predictor_heads
        )

        self.ema_momentum = ema_momentum

    @torch.no_grad()
    def update_target_encoder(self, momentum: float = None):
        """Call after every optimizer step to update target encoder via EMA."""
        tau = momentum if momentum is not None else self.ema_momentum
        update_ema(self.context_encoder, self.target_encoder, tau)

    def forward(
        self,
        images: torch.Tensor,                    # (B, C, H, W)
        context_mask: torch.Tensor,              # 1-D LongTensor, context patch indices
        target_masks: List[torch.Tensor],        # list of 1-D LongTensors per target block
    ) -> Tuple[List[torch.Tensor], List[torch.Tensor]]:
        """
        Returns:
            predictions: List[(B, |tgt_b|, encoder_dim)] — predictor outputs
            targets:     List[(B, |tgt_b|, encoder_dim)] — target encoder outputs (no grad)
        """
        # ── 1. Encode context patches ─────────────────────────────────
        context_emb = self.context_encoder(images, patch_indices=context_mask)
        # (B, |ctx|, encoder_dim)

        # ── 2. Encode ALL target patches with target encoder (no grad) ─
        # Compute target embeddings for the full image first, then index.
        with torch.no_grad():
            all_target_emb = self.target_encoder(images)  # (B, N, encoder_dim)

        # ── 3. Predict + collect targets for each target block ─────────
        predictions, targets = [], []
        for tgt_mask in target_masks:
            tgt_mask = tgt_mask.to(images.device)

            # Predicted embeddings from predictor
            pred = self.predictor(context_emb, context_mask, tgt_mask)
            predictions.append(pred)  # (B, |tgt|, encoder_dim)

            # Ground-truth target embeddings (stop-grad already from no_grad context)
            tgt = all_target_emb[:, tgt_mask, :]           # (B, |tgt|, encoder_dim)
            targets.append(tgt)

        return predictions, targets


# --- Instantiate and verify shapes ---
model = IJEPA(
    img_size=224, patch_size=16, in_chans=3,
    encoder_dim=384, encoder_depth=6, encoder_heads=6,
    predictor_dim=192, predictor_depth=6, predictor_heads=12,
)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable:,}  (target encoder excluded)")

# Forward pass
imgs = torch.randn(2, 3, 224, 224)
ctx_mask, tgt_masks = mask_gen()
preds, tgts = model(imgs, ctx_mask, tgt_masks)
print(f"\nForward pass completed:")
for i, (p, t) in enumerate(zip(preds, tgts)):
    print(f"  Target block {i}: pred={p.shape}  tgt={t.shape}")

## 10. Training Loop

The I-JEPA training loop has one extra step beyond standard training: **EMA update** of the target encoder after each optimizer step.

```python
for images in dataloader:
    1. Sample context + target masks
    2. Forward pass → predictions, targets
    3. Compute JEPA loss
    4. Backward pass + optimizer step  (context encoder + predictor only)
    5. EMA update target encoder
    6. Anneal EMA momentum τ toward 1.0
```

In [ ]:
def train_one_epoch(model, dataloader, optimizer, mask_gen, total_steps, current_step,
                    base_momentum=0.996, final_momentum=1.0, device='cpu'):
    """
    One epoch of I-JEPA training.

    Args:
        model:          IJEPA instance
        dataloader:     yields batches of images (B, C, H, W)
        optimizer:      only for context_encoder + predictor params
        mask_gen:       IJEPAMaskGenerator instance
        total_steps:    total training steps (for EMA schedule)
        current_step:   global step counter
    Returns:
        average loss for this epoch, updated current_step
    """
    model.train()
    epoch_loss = 0.0

    for batch_idx, images in enumerate(dataloader):
        images = images.to(device)
        B = images.shape[0]

        # ── 1. Sample masks (one set of masks per batch) ───────────────
        ctx_mask, tgt_masks = mask_gen()
        ctx_mask = ctx_mask.to(device)
        tgt_masks = [m.to(device) for m in tgt_masks]

        # ── 2. Forward pass ────────────────────────────────────────────
        predictions, targets = model(images, ctx_mask, tgt_masks)

        # ── 3. Compute loss ────────────────────────────────────────────
        loss = jepa_loss(predictions, targets)

        # ── 4. Backward + optimizer step ──────────────────────────────
        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # ── 5. EMA update target encoder ──────────────────────────────
        tau = cosine_ema_schedule(base_momentum, final_momentum, current_step, total_steps)
        model.update_target_encoder(momentum=tau)

        epoch_loss    += loss.item()
        current_step  += 1

        if batch_idx % 10 == 0:
            print(f"  step {current_step:4d}  loss={loss.item():.4f}  τ={tau:.5f}")

    return epoch_loss / len(dataloader), current_step


# --- Smoke test with a tiny fake dataloader ---
class FakeDataLoader:
    """Yields random image tensors — for testing the training loop."""
    def __init__(self, n_batches=5, batch_size=2, img_size=224):
        self.n_batches  = n_batches
        self.batch_size = batch_size
        self.img_size   = img_size
    def __len__(self):  return self.n_batches
    def __iter__(self):
        for _ in range(self.n_batches):
            yield torch.randn(self.batch_size, 3, self.img_size, self.img_size)


# Only optimise context_encoder + predictor (NOT target_encoder)
optimizable_params = [
    {'params': model.context_encoder.parameters()},
    {'params': model.predictor.parameters()},
]
optimizer = torch.optim.AdamW(optimizable_params, lr=1.5e-4, weight_decay=0.05)

fake_loader  = FakeDataLoader(n_batches=3, batch_size=2)
total_steps  = 300 * len(fake_loader)  # simulate 300 epochs

print("Running 1 epoch smoke test ...")
avg_loss, _ = train_one_epoch(
    model, fake_loader, optimizer, mask_gen,
    total_steps=total_steps, current_step=0, device='cpu'
)
print(f"\nAverage loss (untrained, random): {avg_loss:.4f}")

## 11. Extensions: V-JEPA and A-JEPA

The JEPA framework is **modality-agnostic** — the same three-component structure (context encoder, EMA target encoder, predictor) applies to any sequence of tokens.

### V-JEPA (Video)

- Patch tokens come from **space-time tubes** — e.g. 2×16×16 (2 frames × 16×16 pixels)
- Target masks are **temporal blocks**: contiguous spans of frames at random spatial positions
- The predictor must learn to predict the future representation of a region given its past context
- Key difference from I-JEPA: **temporal positional encodings** added alongside spatial ones

```python
# V-JEPA sketch — only the tokenisation changes
class VideoPatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, t_patch_size=2,
                 n_frames=16, in_chans=3, embed_dim=768):
        # 3D Conv: kernel = (t_patch_size, patch_size, patch_size)
        self.projection = nn.Conv3d(
            in_chans, embed_dim,
            kernel_size=(t_patch_size, patch_size, patch_size),
            stride=(t_patch_size, patch_size, patch_size)
        )
        # Separate spatial + temporal positional embeddings (or factorised RoPE)
        self.spatial_pos  = nn.Parameter(torch.zeros(...))
        self.temporal_pos = nn.Parameter(torch.zeros(...))
```

### A-JEPA (Audio)

- Input: log-mel spectrogram tokens (time × frequency patches)
- Target masks: contiguous time spans (the predictor learns to predict the acoustic future)
- Encoder: same ViT architecture (2-D spectrogram patches)

### Data2Vec 2.0

Meta's **Data2Vec 2.0** generalises JEPA across modalities in a single training loop:
- Text: masked word tokens → predict contextualised token representations
- Image: masked image patches → predict ViT encoder representations  
- Audio: masked spectrogram frames → predict wav2vec representations

All three use the same EMA teacher + student student architecture.

In [ ]:
# ── Summary: Full I-JEPA at a Glance ──────────────────────────────────────────

summary = """
I-JEPA Summary
==============

Architecture components:
  1. PatchEmbedding         — splits image into NxN patch tokens
  2. VisionTransformerEncoder (context)  — fθ, trained by backprop
  3. VisionTransformerEncoder (target)   — fξ, updated by EMA only
  4. IJEPAPredictor          — gφ, narrow Transformer, trained by backprop

Training objective:
  L = mean over target blocks of || norm(gφ(fθ(ctx))) - sg(fξ(tgt)) ||²

Key design choices:
  • EMA target encoder     → prevents representation collapse
  • Narrow predictor       → forces encoder to learn rich features
  • Large contiguous masks → encourages semantic reasoning, not texture
  • No negative samples    → simpler training than contrastive methods
  • No augmentation needed → no domain-specific heuristics

Reported results (ViT-H/16 on ImageNet):
  • Linear probing: 81.9% top-1  (vs MAE: 73.5%, DINO: 78.2%)
  • Semi-supervised (1%):  72.5% top-1
  • Semi-supervised (10%): 79.6% top-1
  • Trained on ImageNet-1K only (no extra data)
"""
print(summary)

## 12. Further Reading

| Resource | Link |
|----------|------|
| I-JEPA paper | Assran et al. (2023) — *Self-Supervised Learning from Images with a Joint-Embedding Predictive Architecture* |
| V-JEPA paper | Bardes et al. (2024) — *V-JEPA: Latent Video Prediction for Visual Representation Learning* |
| LeCun's essay | *A Path Towards Autonomous Machine Intelligence* (2022) |
| Official I-JEPA code | [github.com/facebookresearch/ijepa](https://github.com/facebookresearch/ijepa) |
| Official V-JEPA code | [github.com/facebookresearch/jepa](https://github.com/facebookresearch/jepa) |
| BYOL (EMA teacher)   | Grill et al. (2020) — predecessor using EMA for SSL |
| MAE (generative SSL) | He et al. (2022) — compare/contrast with JEPA |

---

### What you built in this notebook

✅ `PatchEmbedding` — tokenise images into patch sequences  
✅ `IJEPAMaskGenerator` — sample context + target block masks  
✅ `VisionTransformerEncoder` — ViT backbone with selective patch masking  
✅ `IJEPAPredictor` — narrow Transformer predicting target embeddings from context  
✅ `update_ema()` + `cosine_ema_schedule()` — EMA target encoder update  
✅ `jepa_loss()` — normalised L2 loss in representation space  
✅ `IJEPA` — full model combining all components  
✅ `train_one_epoch()` — complete training loop with EMA update  

**Next:** See `jepa-02.ipynb` for loading pretrained I-JEPA weights and downstream evaluation (linear probing, fine-tuning on ImageNet).